In [1]:
import os
import sys
from dask.distributed import Client
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json', threads_per_worker=2, n_workers=6)
client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json')  
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler_10.json')  

# add private module path for workers
# client.run(lambda: os.environ.update({'PYTHONPATH': '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'}))
# def add_path():
#     if '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2' not in sys.path:
#         sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')

# client.run(add_path)

def setup_module_path():
    module_path = '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'
    if module_path not in sys.path:
        sys.path.append(module_path)

client.run(setup_module_path)

client

<Client: 'tcp://203.247.189.224:36475' processes=5 threads=90, memory=419.10 GiB>

# Load modules

In [2]:
# load public modules

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
import cmocean
from cmcrameri import cm
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import cftime
import pop_tools
from pprint import pprint
import time
import subprocess
import re as re_mod
import cftime
import datetime
from scipy.stats import ttest_1samp
import xcesm

In [3]:
# load private modules

import sys
sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')
from KYY_CESM2_preprocessing import CESM2_config
# import KYY_CESM2_preprocessing
# import importlib
# importlib.reload(KYY_CESM2_preprocessing)

# Variable configuration

In [4]:

# cfg_var_photoC_TOT_zint=CESM2_config()
# cfg_var_photoC_TOT_zint.year_s=1960
# cfg_var_photoC_TOT_zint.year_e=1964
# cfg_var_photoC_TOT_zint.setvar('photoC_TOT_zint')
# # cfg_var_photoC_TOT_zint.list()


# if cfg_var_photoC_TOT_zint.comp=='ocn':
#     ds_grid = pop_tools.get_grid('POP_gx1v7')

In [67]:
# =====================================================
# Settings
# =====================================================
# varnames = [
#     "photoC_TOT_zint",
#     "SST"
# ]
varnames = [
    "photoC_TOT_zint",
    "SST",
    "SSS2",
    "photoC_TOT",
    "photoC_diat_zint",
    "photoC_diaz_zint",
    "photoC_sp_zint",
    "spC",
    "diatC",
    "diazC",
    "NO3",
    "PO4",
    "TEMP",
    "SALT",
    "Fe",
    "SiO3",
]


# Read files (functions)

In [68]:
# # define preprocessing function

# exceptcv=['time','lon','lat','lev', 'TLONG', 'TLAT', 'z_w', cfg_var_FG_ALT_CO2.var]
# exceptcv = [
#     'time', 'lon', 'lat', 'lev', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'z_t_150m', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
# ] + [cfg.var for cfg in cfgs.values()]
exceptcv = [
    'time', 'lon', 'lat', 'lev', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'z_t_150m', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
] + varnames

def process_coords(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]
    # if comp == "atm" or comp == "lnd":
    #     ds['lat'] = ds['lat'].round(4)
    #     ds['lon'] = ds['lon'].round(4)
    # if comp == "ocn" or comp == "ice":
    #     ds['TLAT'] = ds['TLAT'].round(4)
    #     ds['TLONG'] = ds['TLONG'].round(4)
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        # ds= ds.sel(time=slice(sd, ed)).isel(lev=slice(1, 11))
        return ds
    else:
        return ds.set_coords(coord_vars)


# def process_coords_obs(ds, drop=True, except_coord_vars=['time','lon','lat', 'TLONG', 'TLAT', 'dco2']):
#     """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
#     coord_vars = []
#     for v in np.array(ds.coords) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
#     for v in np.array(ds.data_vars) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
#     if comp == "atm" or comp == "lnd":
#         ds['lat'] = ds['lat'].round(4)
#         ds['lon'] = ds['lon'].round(4)
#     if comp == "ocn" or comp == "ice":
#         ds['TLAT'] = ds['TLAT'].round(4)
#         ds['TLONG'] = ds['TLONG'].round(4)
#     if drop:
#         ds= ds.drop(coord_vars)
#         return ds
#     else:
#         return ds.set_coords(coord_vars)


# def process_coords_obs_fgco2(ds, drop=True, except_coord_vars=['time','lon','lat', 'TLONG', 'TLAT', 'fgco2_smoothed']):
#     """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
#     coord_vars = []
#     for v in np.array(ds.coords) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
#     for v in np.array(ds.data_vars) :
#         if not v in except_coord_vars:
#             coord_vars += [v]
#     if comp == "atm" or comp == "lnd":
#         ds['lat'] = ds['lat'].round(4)
#         ds['lon'] = ds['lon'].round(4)
#     if comp == "ocn" or comp == "ice":
#         ds['TLAT'] = ds['TLAT'].round(4)
#         ds['TLONG'] = ds['TLONG'].round(4)
#     if drop:
#         ds= ds.drop(coord_vars)
#         # ds= ds.sel(time=slice(sd, ed))
#         return ds
#     else:
#         return ds.set_coords(coord_vars)



# Read files (Ocean Data Assimilation)

# Take ensemble means (ADA, ODA, LE, OBS, WDA)

In [69]:
# import os
# import xarray as xr

# varname="photoC_TOT_zint"


# varnames = [
#     "photoC_TOT_zint",
#     "NO3"
# ]

# # cfgs = {}

# cfg = CESM2_config()
# cfg.year_s = 1960
# cfg.year_e = 1964
# cfg.setvar(varname)

# start_time = time.time()

# cfg.ODA_path_load(cfg.var)

# ODA_ds = xr.open_mfdataset(cfg.ODA_file_list[0][0:10],
#                            chunks={'time': 12}, 
#                            combine='nested', 
#                            concat_dim=[[*cfg.ODA_ensembles[0:10]], 'time'], 
#                            parallel=True,
#                            preprocess=lambda ds: process_coords(ds, start_date, end_date),
#                            decode_cf=True, 
#                            decode_times=True)

# ODA_ds2 = xr.open_mfdataset(cfg.ODA_file_list[0][10:20], 
#                            chunks={'time': 12}, 
#                            combine='nested', 
#                            concat_dim=[[*cfg.ODA_ensembles[10:20]], 'time'], 
#                            parallel=True,
#                            preprocess=lambda ds: process_coords(ds, start_date, end_date),
#                            decode_cf=True, 
#                            decode_times=True)

# # en4.2 -> until 2021; projdv7.3 -> until 2020; need to be read separately
# ODA_ds_a = []
# ODA_ds_a.append(ODA_ds)
# ODA_ds_xr = xr.concat(ODA_ds_a, dim='time')
# ODA_ds_xr = ODA_ds_xr.expand_dims({'ens': range(10)})

# ODA_ds2_a = []
# ODA_ds2_a.append(ODA_ds2)
# ODA_ds2_xr = xr.concat(ODA_ds2_a, dim='time')
# ODA_ds2_xr = ODA_ds2_xr.expand_dims({'ens': range(10, 20)})

# cfg.ODA_ds = []
# cfg.ODA_ds.append(ODA_ds_xr)
# cfg.ODA_ds.append(ODA_ds2_xr)
# cfg.ODA_ds = xr.concat(cfg.ODA_ds, dim='ens')
# cfg.ODA_ds = cfg.ODA_ds.reindex(ens=list(range(1, 20+1)))
# cfg.ODA_ds = cfg.ODA_ds.sortby('time')

# cfg.ODA_ds = cfg.ODA_ds.rename({"concat_dim": "ens_ODA"})
# new_time = cfg.ODA_ds.time - np.array([datetime.timedelta(days=15)] * len(cfg.ODA_ds.time))
# cfg.ODA_ds = cfg.ODA_ds.assign_coords(time=new_time)
# cfg.ODA_ds = cfg.ODA_ds.drop('ens')
# cfg.ODA_ds=cfg.ODA_ds.mean(dim='ens')
# end_time = time.time()


# # =====================================================
# # Base output directory
# # =====================================================
# BASE_OUTDIR = "/mnt/lustre/proj/earth.system.predictability/ASSM_EXP_timeseries/archive_ensmean"

# comp  = cfg.comp    # 예: "ocn"
# tfreq = cfg.tfreq   # 예: "month_1"

# outdir = os.path.join(BASE_OUTDIR, comp, tfreq)
# os.makedirs(outdir, exist_ok=True)

# # =====================================================
# # Output filename
# # =====================================================
# outfile = os.path.join(
#     outdir,
#     f"b.e21.BHISTsmbb.f09_g17.assm.ENSMEAN.pop.h."
#     f"{varname}.196001-196412.nc"
# )


# # =====================================================
# # 1. Load dataset
# # =====================================================
# ds = cfg.ODA_ds

# # =====================================================
# # 2. Ensemble mean
# # =====================================================
# ds_ensmean = ds.mean(dim="ens_ODA", keep_attrs=True)


# # =====================================================
# # 3. Build output dataset (POP-like structure)
# # =====================================================
# ds_out = xr.Dataset(
#     data_vars={
#         "photoC_TOT_zint": ds_ensmean["photoC_TOT_zint"]
#     },
#     coords={
#         "time": ds_ensmean["time"],
#         "nlat": ds_ensmean["nlat"],
#         "nlon": ds_ensmean["nlon"],
#         "TLAT": ds_ensmean["TLAT"],
#         "TLONG": ds_ensmean["TLONG"],
#     },
#     attrs=ds.attrs
# )

# # copy variable attributes
# ds_out["photoC_TOT_zint"].attrs = ds["photoC_TOT_zint"].attrs

# # =====================================================
# # 4. Encoding (POP-style)
# # =====================================================
# encoding = {
#     "photoC_TOT_zint": {
#         "dtype": "float32",
#         "zlib": True,
#         "complevel": 4,
#         "_FillValue": 9.96921e36,
#     }
# }

# # =====================================================
# # 5. Save netCDF
# # =====================================================

# ds_out.to_netcdf(
#     outfile,
#     format="NETCDF4_CLASSIC",
#     encoding=encoding
# )

# elapsed_time = end_time - start_time
# print('elasped time for reading ODA' + varname + ': ' + str(elapsed_time))

In [70]:
# import os
# import time
# import datetime
# import numpy as np
# import xarray as xr

# # =====================================================
# # settings
# # =====================================================
# # varnames = [
# #     "photoC_TOT_zint",
# #     "SST",
# #     # 추가 가능
# # ]

# BASE_OUTDIR = "/mnt/lustre/proj/earth.system.predictability/ASSM_EXP_timeseries/archive_ensmean"

# for varname in varnames:

#     cfg = CESM2_config()
#     cfg.year_s = 1960
#     cfg.year_e = 1964
#     cfg.setvar(varname)

#     start_time = time.time()

#     cfg.ODA_path_load(cfg.var)

#     # =================================================
#     # read (split, EXACTLY as original)
#     # =================================================
#     # =========================
#     # read (two blocks)
#     # =========================
#     ODA_ds1 = xr.open_mfdataset(
#         cfg.ODA_file_list[0][0:10],
#         chunks={"time": 12},
#         combine="nested",
#         concat_dim=[[*cfg.ODA_ensembles[0:10]], 'time'], 
#         parallel=True,
#         preprocess=lambda ds: process_coords(ds, start_date, end_date),
#         decode_cf=True,
#         decode_times=True,
#     )
    
#     ODA_ds2 = xr.open_mfdataset(
#         cfg.ODA_file_list[0][10:20],
#         chunks={"time": 12},
#         combine="nested",
#         concat_dim=[[*cfg.ODA_ensembles[10:20]], 'time'], 
#         parallel=True,
#         preprocess=lambda ds: process_coords(ds, start_date, end_date),
#         decode_cf=True,
#         decode_times=True,
#     )
    
#     # =========================
#     # concat two parts along concat_dim
#     # =========================
#     ds = xr.concat([ODA_ds1, ODA_ds2], dim="concat_dim")
    
#     # =========================
#     # rename concat_dim -> ens_ODA
#     # =========================
#     ds = ds.rename({"concat_dim": "ens_ODA"})
    
#     # =========================
#     # time shift (기존 로직)
#     # =========================
#     ds = ds.assign_coords(
#         time=ds.time - np.array([datetime.timedelta(days=15)] * ds.sizes["time"])
#     )
    
#     # =========================
#     # ensemble mean
#     # =========================
#     ds = ds.mean(dim="ens_ODA", keep_attrs=True)
    
#     cfg.ODA_ds = ds

#     end_time = time.time()

#     # =================================================
#     # output path
#     # =================================================
#     outdir = os.path.join(BASE_OUTDIR, cfg.comp, cfg.tfreq)
#     os.makedirs(outdir, exist_ok=True)

#     outfile = os.path.join(
#         outdir,
#         f"b.e21.BHISTsmbb.f09_g17.assm.ENSMEAN.{cfg.model}."
#         f"{varname}.196001-196412.nc"
#     )

#     # =================================================
#     # save (coords 그대로)
#     # =================================================
#     ds_out = cfg.ODA_ds[[varname]]

#     encoding = {
#         varname: {
#             "dtype": "float32",
#             "zlib": True,
#             "complevel": 4,
#             "_FillValue": 9.96921e36,
#         }
#     }

#     ds_out.to_netcdf(
#         outfile,
#         format="NETCDF4_CLASSIC",
#         encoding=encoding,
#     )

#     print(
#         f"{varname} done | elapsed: {end_time - start_time:.1f} s"
#     )


In [71]:
import os
import time
import datetime
import numpy as np
import xarray as xr

# =====================================================
# settings
# =====================================================
BASE_OUTDIR = "/mnt/lustre/proj/earth.system.predictability/ASSM_EXP_timeseries/archive_ensmean"

# 5-year blocks + last single year
year_blocks = [(y, y+4) for y in range(1960, 2020, 5)] + [(2020, 2020)]

for varname in varnames:
    for year_s, year_e in year_blocks:

        cfg = CESM2_config()
        cfg.year_s = year_s
        cfg.year_e = year_e
        cfg.setvar(varname)

        start_time = time.time()

        cfg.ODA_path_load(cfg.var)

        start_date = cftime.DatetimeNoLeap(cfg.year_s, 2, 1)
        end_date = cftime.DatetimeNoLeap(cfg.year_e+1, 1, 1)
        
        # =========================
        # read (two blocks)
        # =========================
        ODA_ds1 = xr.open_mfdataset(
            cfg.ODA_file_list[0][0:10],
            chunks={"time": 12},
            combine="nested",
            concat_dim=[[*cfg.ODA_ensembles[0:10]], "time"],
            parallel=True,
            preprocess=lambda ds: process_coords(ds, start_date, end_date),
            decode_cf=True,
            decode_times=True,
        )

        ODA_ds2 = xr.open_mfdataset(
            cfg.ODA_file_list[0][10:20],
            chunks={"time": 12},
            combine="nested",
            concat_dim=[[*cfg.ODA_ensembles[10:20]], "time"],
            parallel=True,
            preprocess=lambda ds: process_coords(ds, start_date, end_date),
            decode_cf=True,
            decode_times=True,
        )

        # =========================
        # concat + rename
        # =========================
        ds = xr.concat([ODA_ds1, ODA_ds2], dim="concat_dim")
        ds = ds.rename({"concat_dim": "ens_ODA"})

        # time shift
        ds = ds.assign_coords(
            time=ds.time - np.array([datetime.timedelta(days=15)] * ds.sizes["time"])
        )

        # ensemble mean
        cfg.ODA_ds = ds.mean(dim="ens_ODA", keep_attrs=True)

        end_time = time.time()

        # =========================
        # output
        # =========================
        outdir = os.path.join(BASE_OUTDIR, cfg.comp, cfg.tfreq)
        os.makedirs(outdir, exist_ok=True)

        if year_s == year_e:
            period = f"{year_s}01-{year_e}12"
        else:
            period = f"{year_s}01-{year_e}12"

        outfile = os.path.join(
            outdir,
            f"b.e21.BHISTsmbb.f09_g17.assm.ENSMEAN.{cfg.model}."
            f"{varname}.{period}.nc"
        )

        ds_out = cfg.ODA_ds[[varname]]

        encoding = {
            varname: {
                "dtype": "float32",
                "zlib": True,
                "complevel": 4,
                "_FillValue": 9.96921e36,
            }
        }

        ds_out.to_netcdf(
            outfile,
            format="NETCDF4_CLASSIC",
            encoding=encoding,
        )

        print(
            f"{varname} {year_s}-{year_e} done | elapsed: {end_time - start_time:.1f} s"
        )


photoC_TOT_zint 1960-1964 done | elapsed: 43.8 s
photoC_TOT_zint 1965-1969 done | elapsed: 20.6 s
photoC_TOT_zint 1970-1974 done | elapsed: 18.2 s
photoC_TOT_zint 1975-1979 done | elapsed: 20.6 s
photoC_TOT_zint 1980-1984 done | elapsed: 18.9 s
photoC_TOT_zint 1985-1989 done | elapsed: 21.1 s
photoC_TOT_zint 1990-1994 done | elapsed: 23.1 s
photoC_TOT_zint 1995-1999 done | elapsed: 21.8 s
photoC_TOT_zint 2000-2004 done | elapsed: 18.0 s
photoC_TOT_zint 2005-2009 done | elapsed: 18.4 s
photoC_TOT_zint 2010-2014 done | elapsed: 23.7 s
photoC_TOT_zint 2015-2019 done | elapsed: 22.0 s
photoC_TOT_zint 2020-2020 done | elapsed: 11.8 s
SST 1960-1964 done | elapsed: 5.4 s
SST 1965-1969 done | elapsed: 1.6 s
SST 1970-1974 done | elapsed: 1.9 s
SST 1975-1979 done | elapsed: 2.1 s
SST 1980-1984 done | elapsed: 2.0 s
SST 1985-1989 done | elapsed: 2.2 s
SST 1990-1994 done | elapsed: 2.0 s
SST 1995-1999 done | elapsed: 2.5 s
SST 2000-2004 done | elapsed: 2.0 s
SST 2005-2009 done | elapsed: 1.6 s
SST